# Process-Level Visualization Demo

This notebook demonstrates how to visualize process-level metrics across multiple batches in AnimaKinetica.

## Features:
- Plot cumulative biomass and volume harvest
- Visualize cumulative medium consumption
- Compare state variable profiles across batches
- Analyze long-term process performance

---

In [1]:
# Import necessary modules
import numpy as np
from animakinetica.core.parameters import SimulationParameters
from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.analysis.process_visualization import (
    plot_simulation_cumulative_harvest,
    plot_simulation_cumulative_medium_use,
    plot_simulation_profiles
)
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.perfusion_strategies import PerfusionStrategy

## 1. Initialize Simulation

Load parameters and create a simulation instance.

In [2]:
# Load default parameters
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "fed_batch"])

# Create simulation with STOICHIOMETRIC kinetics and FEDBATCH mode
sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    feed_strategy=FeedStrategy.EXPONENTIAL,
    operating_mode=OperatingMode.FEDBATCH
)

print(f"Simulation Type: {sim.data.simulation_type}")

Loading parameters from 5 file(s) sequentially...
  [1/5] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\operating_modes\fed_batch.yaml
Successfully loaded parameters from 5 file(s).
Simulation Type: 


## 2. Run Multiple Batches

Execute 10 batches to collect process-level data.

In [3]:
# Run 10 batches
results = sim.run_simulation(number_of_batches=10)

print(f"\nCompleted {len(sim.data.batches)} batches")
print(f"\nBatch Summary:")
for i, batch in enumerate(sim.data.batches, 1):
    print(f"Batch {i}: {batch.t_end:.1f} h, Final Xv: {batch.Xv_final/1e9:.2f} x 10^6 cells/mL, Final V: {batch.V_final:.1f} L")

run_simulation execution time: 326.400 ms

Completed 10 batches

Batch Summary:
Batch 1: 144.0 h, Final Xv: 9.82 x 10^6 cells/mL, Final V: 2000.0 L
Batch 2: 107.1 h, Final Xv: 9.86 x 10^6 cells/mL, Final V: 2000.0 L
Batch 3: 107.0 h, Final Xv: 9.83 x 10^6 cells/mL, Final V: 2000.0 L
Batch 4: 106.9 h, Final Xv: 9.82 x 10^6 cells/mL, Final V: 2000.0 L
Batch 5: 106.9 h, Final Xv: 9.82 x 10^6 cells/mL, Final V: 2000.0 L
Batch 6: 106.9 h, Final Xv: 9.82 x 10^6 cells/mL, Final V: 2000.0 L
Batch 7: 106.9 h, Final Xv: 9.82 x 10^6 cells/mL, Final V: 2000.0 L
Batch 8: 106.9 h, Final Xv: 9.82 x 10^6 cells/mL, Final V: 2000.0 L
Batch 9: 67.3 h, Final Xv: 10.43 x 10^6 cells/mL, Final V: 740.1 L
Batch 10: 144.0 h, Final Xv: 9.82 x 10^6 cells/mL, Final V: 2000.0 L


## 3. Plot Cumulative Harvest

Visualize cumulative biomass and volume harvested over time.

In [4]:
# Plot cumulative harvest for all batches
fig_harvest = plot_simulation_cumulative_harvest(sim, max_batches=10)
fig_harvest.show()

# Alternative: using the data object method
# fig_harvest = sim.data.plot_cumulative_harvest(max_batches=10)
# fig_harvest.show()

### Interpretation:

**Cumulative Biomass (primary y-axis, red):**
- Total biomass harvested in kg
- Stepwise increases at end of each batch
- Slope indicates harvest rate
- Key metric for productivity assessment

**Cumulative Volume (secondary y-axis, purple):**
- Total liquid volume harvested in L
- Includes harvest from fedbatch/perfusion operations
- Important for downstream processing capacity planning

**Use for:**
- Production planning
- Capacity utilization analysis
- Yield calculation over campaign duration

## 4. Plot Cumulative Medium Use

Visualize total medium consumption (batch + feed).

In [5]:
# Plot cumulative medium use for all batches
fig_medium = plot_simulation_cumulative_medium_use(sim, max_batches=10)
fig_medium.show()

# Alternative: using the data object method
# fig_medium = sim.data.plot_cumulative_medium_use(max_batches=10)
# fig_medium.show()

### Interpretation:

**Total Medium (green):**
- Sum of batch + feed medium
- Total liquid raw material cost
- Critical for process economics

**Feed Medium (orange):**
- Medium added during batch operation
- Only present in Fedbatch, Perfusion, Concentrated Fedbatch modes
- Zero in Batch mode

**Batch Medium (blue):**
- Initial medium at batch start
- Stepwise increase at each new batch
- Constant within each batch

**Use for:**
- Raw material cost analysis
- Inventory management
- Comparing feeding strategies
- Yield per liter medium (productivity/cost metric)

## 3. Test plot_simulation_cumulative_base_use

Test the new cumulative base usage visualization function.

### 3.1. Import the new function

In [6]:
from animakinetica.analysis.process_visualization import plot_simulation_cumulative_base_use

### 3.2. Plot cumulative base use from simulation

In [7]:
# Test with SimulationData
fig_base = plot_simulation_cumulative_base_use(results, max_batches=5)
fig_base.show()

**Interpretation:**
- Shows cumulative base (NaOH) consumption across batches
- Calculated as: D_base × V × dt integrated over time
- Units in liters (L) for easy cost and operational analysis
- Important for:
  - Economic analysis (base is a consumable)
  - Process design (sizing base storage tanks)
  - Understanding pH control requirements
  - Scale-up considerations

### 3.3. Test with custom layout

In [8]:
# Test with custom layout
custom_layout = {'height': 500, 'width': 1000}
fig_base_custom = plot_simulation_cumulative_base_use(results, max_batches=5, layout_kwargs=custom_layout)
fig_base_custom.show()

## 4. Plot State Variable Profiles

Overlay profiles from multiple batches to identify trends.

In [9]:
# Plot profiles for first 5 batches
figs = plot_simulation_profiles(sim, max_batches=5)
# figs is a list of up to 9 figures (depending on data availability)
# List of 12 interactive figures (cell density, glucose, glutamine, lactate, ammonia,
# amino acids, DO, C_CT, pH, C_CO2, osmolality, Na⁺)
# [0] Cell Density
# [1] Glucose
# [2] Glutamine
# [3] Lactate
# [4] Ammonia
# [5] Amino Acids
# [6] Dissolved O2 (DO)
# [7] Total Dissolved Carbon (C_CT)
# [8] pH
# [9] Dissolved CO2
# [10] Osmolality
# [11] Sodium (Na⁺)

print(f"Generated {len(figs)} profile figures")

Generated 12 profile figures


### 4.1. Cell Density Profile

In [10]:
# Display cell density profile
figs[0].show()

**Interpretation:**
- Each colored line = one batch
- Time axis is continuous (batches placed sequentially)
- Identify batch-to-batch variability
- Look for drift in final cell density

### 4.2. Glucose Profile

In [11]:
# Display glucose profile
figs[1].show()

**Interpretation:**
- Shows glucose depletion over each batch
- Compare depletion rates across batches
- Identify if feeding strategy maintains adequate glucose levels

### 4.3. Glutamine Profile

In [12]:
# Display glutamine profile
figs[2].show()

**Interpretation:**
- Glutamine is often limiting before glucose
- Look for early depletion patterns
- Consider glutamine supplementation if consistently depleted early

### 4.4. Lactate Profile

In [13]:
# Display lactate profile
figs[3].show()

**Interpretation:**
- Lactate accumulates as waste product
- High levels inhibit growth
- Compare final lactate levels across batches
- Lower lactate = better metabolic efficiency

### 4.5. Ammonia Profile

In [14]:
# Display ammonia profile
figs[4].show()

**Interpretation:**
- Ammonia from glutamine degradation
- Toxic at high concentrations
- Monitor final levels for inhibition thresholds

### 4.6 Amino Acids (AA) Profile

In [15]:
figs[5].show()

### 4.7. Dissolved Oxygen (DO) Profile

In [16]:
# Display dissolved oxygen profile (if available)
if len(figs) > 6:
    figs[6].show()
else:
    print("Dissolved oxygen data not available in this simulation")

**Interpretation:**
- Dissolved oxygen concentration in liquid phase
- Critical for aerobic metabolism
- Should be maintained above critical DO for optimal growth
- Low DO indicates oxygen limitation

### 4.8. Total Dissolved Carbon (C_CT) Profile

In [17]:
# Display total dissolved carbon profile (if available)
if len(figs) > 7:
    figs[7].show()
else:
    print("Total dissolved carbon data not available in this simulation")

**Interpretation:**
- Sum of all dissolved carbon species (CO2, HCO3-, CO3--)
- Increases with CO2 production during growth
- Related to pH and CO2 accumulation
- Important for carbonate buffering system

### 4.9. pH Profile

In [18]:
# Display pH profile (if available)
if len(figs) > 8:
    figs[8].show()
else:
    print("pH data not available in this simulation")

**Interpretation:**
- Converted from H+ concentration: pH = -log10([H+])
- Critical parameter for cell viability and productivity
- Affected by lactate production, CO2 accumulation, and base addition
- Should be maintained within optimal range (typically 6.8-7.4)
- pH control via base addition prevents drift

### 4.10. Dissolved CO₂ Profile

In [19]:
# Display dissolved CO2 profile (if available)
if len(figs) > 9:
    figs[9].show()
else:
    print("Dissolved CO2 data not available in this simulation")

**Interpretation:**
- Dissolved CO2 concentration in liquid phase
- Produced by cellular respiration
- High levels can inhibit cell growth and productivity
- Gas sparging and mass transfer help remove excess CO2
- Important for scale-up considerations (harder to remove at larger scales)

### 4.11. Osmolality Profile

In [20]:
# Display osmolality profile (if available)
if len(figs) > 10:
    figs[10].show()
else:
    print("Osmolality data not available in this simulation")

**Interpretation:**
- Total osmolality of the culture medium (mOsm/kg)
- Increases with feed addition and metabolite accumulation
- High osmolality can stress cells and reduce viability
- Optimal range typically 280-350 mOsm/kg for mammalian cells
- Important for medium design and feeding strategies

### 4.12. Sodium Concentration (Na⁺) Profile

In [21]:
# Display sodium concentration profile (if available)
if len(figs) > 11:
    figs[11].show()
else:
    print("Sodium concentration data not available in this simulation")

**Interpretation:**
- Sodium ion concentration in the medium (mmol/L)
- Increases with base (NaOH) addition for pH control
- Contributes to osmolality
- Important for ionic balance and cell membrane potential
- High concentrations can contribute to osmotic stress

## 5. Plot all possible metrics

Visualize key process metrics.

In [22]:
figure_dict = sim.data.plot_all(individual_batch=5, max_batches_to_plot=10)

for fig in figure_dict.values():
    fig.show()


Generating all plots for Fedbatch...

[1/16] Plotting batch 5: Biomass & Metabolites...
[2/16] Plotting batch 5: Volume & Feed...
[3/16] Plotting batch 5: Gas Transfer...
[4/16] Plotting batch 5: Nutrient Factors...
[5/16] Plotting batch 5: Inhibition Factors...
[6/16] Plotting batch 5: Kinetic Rates...
[7/16] Plotting batch 5: Complete Kinetics Overview...
[8/16] Plotting batch 5: Stoichiometric Yields...
[9/16] Plotting cumulative harvest...
[10/16] Plotting cumulative medium use...
[11/16] Plotting cumulative gas use...
[12-16/16] Plotting simulation profiles...

✓ All 16 plots generated successfully!



## 6. Compare First vs Last Batch

Identify process drift over campaign duration.

In [23]:
# Compare first and last batch
first_batch = sim.data.batches[0]
last_batch = sim.data.batches[-1]

print("First vs Last Batch Comparison:")
print(f"\nBatch Duration:")
print(f"  First: {first_batch.t_end:.1f} h")
print(f"  Last:  {last_batch.t_end:.1f} h")
print(f"  Change: {((last_batch.t_end - first_batch.t_end)/first_batch.t_end)*100:+.1f}%")

print(f"\nFinal Cell Density:")
print(f"  First: {first_batch.Xv_final/1e9:.2f} x 10^6 cells/mL")
print(f"  Last:  {last_batch.Xv_final/1e9:.2f} x 10^6 cells/mL")
print(f"  Change: {((last_batch.Xv_final - first_batch.Xv_final)/first_batch.Xv_final)*100:+.1f}%")

print(f"\nFinal Volume:")
print(f"  First: {first_batch.V_final:.1f} L")
print(f"  Last:  {last_batch.V_final:.1f} L")
print(f"  Change: {((last_batch.V_final - first_batch.V_final)/first_batch.V_final)*100:+.1f}%")

First vs Last Batch Comparison:

Batch Duration:
  First: 144.0 h
  Last:  144.0 h
  Change: +0.0%

Final Cell Density:
  First: 9.82 x 10^6 cells/mL
  Last:  9.82 x 10^6 cells/mL
  Change: +0.0%

Final Volume:
  First: 2000.0 L
  Last:  2000.0 L
  Change: +0.0%


## 7. Custom Layout Options

Demonstrate plot customization for publication.

In [24]:
# Custom layout for cumulative harvest
custom_layout = {
    'height': 500,
    'width': 1000,
    'font': dict(size=14, family='Arial'),
    'title_font': dict(size=18),
    'template': 'simple_white'
}

fig_custom_harvest = plot_simulation_cumulative_harvest(
    sim,
    max_batches=10,
    layout_kwargs=custom_layout
)
fig_custom_harvest.show()

fig_custom_medium = plot_simulation_cumulative_medium_use(
    sim,
    max_batches=10,
    layout_kwargs=custom_layout
)
fig_custom_medium.show()

## 8. Test with Different Operating Modes

Compare process-level metrics across operating modes.

### 8.1. Batch Mode

In [25]:
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI"])
# Run batch mode simulation
sim_batch = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    feed_strategy=None,
    operating_mode=OperatingMode.BATCH
)

results_batch = sim_batch.run_simulation(number_of_batches=5)

# Plot cumulative harvest
fig_batch_harvest = plot_simulation_cumulative_harvest(sim_batch, max_batches=5)
fig_batch_harvest.show()

# Plot cumulative medium
fig_batch_medium = plot_simulation_cumulative_medium_use(sim_batch, max_batches=5)
fig_batch_medium.show()

Loading parameters from 4 file(s) sequentially...
  [1/4] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/4] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/4] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/4] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
Successfully loaded parameters from 4 file(s).
run_simulation execution time: 124.367 ms


**Note for Batch Mode:**
- No feed medium (orange line should be flat at zero)
- Only batch medium contributes to total
- Typically lower productivity than Fedbatch

### 8.2. Perfusion Mode

In [26]:
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "fed_batch","perfusion"])
# Run perfusion mode simulation
sim_fedbatch = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    feed_strategy=FeedStrategy.EXPONENTIAL,
    perfusion_strategy=PerfusionStrategy.CSPR,
    operating_mode=OperatingMode.CONTINUOUS_PERFUSION
)

results_perfusion = sim_fedbatch.run_simulation(number_of_batches=5)

# Plot cumulative harvest
fig_perfusion_harvest = plot_simulation_cumulative_harvest(sim_fedbatch, max_batches=5)
fig_perfusion_harvest.show()

# Plot cumulative medium
fig_perfusion_medium = plot_simulation_cumulative_medium_use(sim_fedbatch, max_batches=5)
fig_perfusion_medium.show()

Loading parameters from 6 file(s) sequentially...
  [1/6] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\operating_modes\fed_batch.yaml
  [6/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code

**Note for Perfusion Mode:**
- High feed medium usage (continuous feeding)
- Continuous harvest throughout batch
- High medium consumption but also high productivity

### 8.3 Fed-batch mode

In [27]:
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "fed_batch"])
# Run perfusion mode simulation
sim_fedbatch = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    feed_strategy=FeedStrategy.EXPONENTIAL,
    operating_mode=OperatingMode.FEDBATCH
)

results_perfusion = sim_fedbatch.run_simulation(number_of_batches=5)

# Plot cumulative harvest
fig_fedbatch_harvest = plot_simulation_cumulative_harvest(sim_fedbatch, max_batches=5)
fig_fedbatch_harvest.show()

# Plot cumulative medium
fig_fedbatch_medium = plot_simulation_cumulative_medium_use(sim_fedbatch, max_batches=5)
fig_fedbatch_medium.show()

Loading parameters from 5 file(s) sequentially...
  [1/5] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\operating_modes\fed_batch.yaml
Successfully loaded parameters from 5 file(s).
run_simulation execution time: 146.475 ms
